# CEG 图像生成产物 Colab Notebook

该 Notebook 的正式运行流程是:

1. 在 Colab 中运行 Notebook。
2. 从 GitHub 拉取或更新 `CEG` 仓库代码。
3. 如果图像生成需要前序产物, 从 Google Drive 的 `CEG` 工作区加载前序产物结果。
4. 在 Colab GPU 环境中运行仓库脚本和真实外部 backend。
5. 调用真实 SD / watermark backend 生成 clean / watermarked 图像和 image manifests。
6. 运行图像生成产物验收脚本。
7. 刷新阶段摘要。
8. 将图像生成产物打包为 zip, 保存回 Google Drive 的 `CEG` 目录下。

重要边界:
- GitHub 是代码来源, Google Drive 是正式输入、输出和归档位置。
- Notebook 只做环境准备和脚本编排, 不直接手写正式 `prompt_plan.json`、`image_pairs.json` 或 image manifests。
- 真正的图像生成由你配置的外部 SD / watermark backend 完成。
- 图像生成产物是否完成只以 `validate_pilot_image_generation_outputs.py --require-pass` 的结果为准。
- 不要把 Hugging Face token 写入 Notebook、JSON、CSV、manifest 或日志。token 应只存在于 Colab 环境变量或 Colab secret 中。


In [ ]:
# 1. 配置区
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from zipfile import ZipFile, ZIP_DEFLATED

# GitHub 是代码来源。Notebook 会先 clone 或 pull 该仓库, 再读取 Google Drive 中的前序产物。
REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
UPDATE_REPO_FROM_GITHUB = True
INSTALL_REPO_EDITABLE = True

INSTALL_IMAGE_GENERATION_DEPENDENCIES = True

ATTESTATION_KEY_ENV = "CEG_ATTESTATION_KEY"
ATTESTATION_KEY_ID = "formal_colab_run_key"
SEMANTIC_MASK_BACKEND = "ceg_inspyrenet_semantic_mask"
PREPARE_INSPYRENET_WEIGHT = True
INSPYRENET_WEIGHT_URL = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"

DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
PILOT_WORKSPACE_ROOT = DRIVE_ROOT / "pilot_runs" / "real_pilot_input_workspace_20260617_034500"

COMMAND_FILE = PILOT_WORKSPACE_ROOT / "configs" / "image_generation_backend_command.draft.json"
COMMAND_VALIDATION_REPORT = PILOT_WORKSPACE_ROOT / "configs" / "image_generation_backend_command_validation_report.json"
IMAGE_OUTPUT_ROOT = PILOT_WORKSPACE_ROOT / "inputs" / "images"
PROMPT_PLAN = PILOT_WORKSPACE_ROOT / "inputs" / "prompts" / "prompt_plan.draft.json"
MODEL_CONFIG = PILOT_WORKSPACE_ROOT / "configs" / "model_config.draft.json"
INSPYRENET_WEIGHT_DRIVE_PATH = DRIVE_ROOT / "Models" / "inspyrenet" / "ckpt_base.pth"
INSPYRENET_CACHE_PATHS = [Path.home() / ".transparent-background" / "ckpt_base.pth", Path.home() / ".cache" / "transparent-background" / "ckpt_base.pth"]

# 若你已经手工把 COMMAND_FILE 中 external_command_placeholder 替换为 external_command, 保持 False。
# 若希望本 Notebook 写入 external_command, 设置为 True 并填写 EXTERNAL_COMMAND。
APPLY_EXTERNAL_COMMAND_FROM_NOTEBOOK = True
EXTERNAL_COMMAND = [
    "python",
    str(REPO_ROOT / "scripts" / "run_pilot_real_image_generation_backend.py"),
    "--prompt-plan",
    str(PROMPT_PLAN),
    "--out",
    str(IMAGE_OUTPUT_ROOT),
    "--model-config",
    str(MODEL_CONFIG),
    "--watermark-backend",
    "ceg_content_chain_embedding",
    "--content-mask-backend",
    SEMANTIC_MASK_BACKEND,
    "--attestation-key-env",
    ATTESTATION_KEY_ENV,
    "--attestation-key-id",
    ATTESTATION_KEY_ID,
    "--disable-progress-bar",
    "--require-pass",
]

# 运行图像生成包装入口。只有命令文件校验通过后才应设置为 True。
RUN_IMAGE_GENERATION_OUTPUTS = True

# 正式运行必须要求 backend 命令文件通过校验。
REQUIRE_BACKEND_COMMAND_READY = True

# 图像生成产物验收成功后可生成后续接续计划。该步骤不运行 attack 或 detection, 只写 runbook。
BUILD_IMAGE_GENERATION_RESUME_PLAN = True

# 严格检查 Google Drive 工作区。正式运行建议保持 True, 避免把结果写到错误目录。
STRICT_GOOGLE_DRIVE_PREFLIGHT = True

# 图像生成验收通过后, 将关键结果打包为 zip 保存到 Google Drive 的 CEG 目录下。
ARCHIVE_IMAGE_GENERATION_OUTPUTS = True
ARCHIVE_ROOT = DRIVE_ROOT / "archives" / "image_generation_outputs"
ARCHIVE_PATH = ARCHIVE_ROOT / f"{PILOT_WORKSPACE_ROOT.name}_image_generation_outputs.zip"


In [ ]:
# 2. 挂载 Google Drive, 拉取 GitHub 代码, 并准备 Colab 运行环境
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if UPDATE_REPO_FROM_GITHUB:
    if not REPO_URL:
        raise ValueError("UPDATE_REPO_FROM_GITHUB = True 时必须填写 REPO_URL。")
    if not REPO_ROOT.exists():
        clone_cmd = ["git", "clone"]
        if REPO_BRANCH:
            clone_cmd += ["--branch", REPO_BRANCH]
        clone_cmd += [REPO_URL, str(REPO_ROOT)]
        print("运行:", " ".join(clone_cmd))
        subprocess.run(clone_cmd, check=True)
    elif (REPO_ROOT / ".git").exists():
        fetch_cmd = ["git", "-C", str(REPO_ROOT), "fetch", "--all", "--prune"]
        print("运行:", " ".join(fetch_cmd))
        subprocess.run(fetch_cmd, check=True)
        if REPO_BRANCH:
            checkout_cmd = ["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH]
            print("运行:", " ".join(checkout_cmd))
            subprocess.run(checkout_cmd, check=True)
        pull_cmd = ["git", "-C", str(REPO_ROOT), "pull", "--ff-only"]
        print("运行:", " ".join(pull_cmd))
        subprocess.run(pull_cmd, check=True)
    else:
        raise FileNotFoundError(f"REPO_ROOT 已存在但不是 Git 仓库: {REPO_ROOT}")

if not REPO_ROOT.exists():
    raise FileNotFoundError(f"未找到仓库目录: {REPO_ROOT}")

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

if INSTALL_REPO_EDITABLE:
    install_cmd = [sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)]
    print("运行:", " ".join(install_cmd))
    subprocess.run(install_cmd, check=True)

if INSTALL_IMAGE_GENERATION_DEPENDENCIES:
    install_cmd = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "diffusers",
        "transformers",
        "accelerate",
        "safetensors",
        "Pillow",
        "PyYAML",
        "transparent-background",
    ]
    print("运行:", " ".join(install_cmd))
    subprocess.run(install_cmd, check=True)



def ensure_ceg_attestation_key() -> None:
    """在 Colab 运行时定义 CEG_ATTESTATION_KEY, 只放入环境变量, 不写入文件或日志。"""
    if os.environ.get(ATTESTATION_KEY_ENV):
        print(f"{ATTESTATION_KEY_ENV} 已存在, 不打印密钥内容。")
        return
    secret_value = None
    try:
        from google.colab import userdata  # type: ignore
        secret_value = userdata.get(ATTESTATION_KEY_ENV)
    except Exception:
        secret_value = None
    if not secret_value:
        import secrets
        secret_value = secrets.token_urlsafe(48)
        print(f"{ATTESTATION_KEY_ENV} 未在 Colab secrets 中定义, 已为本次运行生成临时密钥。")
    os.environ[ATTESTATION_KEY_ENV] = secret_value
    print(f"{ATTESTATION_KEY_ENV} configured =", bool(os.environ.get(ATTESTATION_KEY_ENV)))


def prepare_inspyrenet_weight() -> None:
    """准备 InSPyReNet 权重, 该逻辑仅属于 Colab 环境准备, 不进入 CEG 主方法。"""
    if not PREPARE_INSPYRENET_WEIGHT:
        print("PREPARE_INSPYRENET_WEIGHT = False, 跳过 InSPyReNet 权重准备。")
        return
    source_path = INSPYRENET_WEIGHT_DRIVE_PATH
    if not source_path.is_file():
        source_path.parent.mkdir(parents=True, exist_ok=True)
        download_cmd = ["python", "-c", "import urllib.request, sys; urllib.request.urlretrieve(sys.argv[1], sys.argv[2])", INSPYRENET_WEIGHT_URL, str(source_path)]
        print("Drive 中未找到 InSPyReNet 权重, 下载到:", source_path)
        subprocess.run(download_cmd, check=True)
    for cache_path in INSPYRENET_CACHE_PATHS:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if not cache_path.exists():
            shutil.copyfile(source_path, cache_path)
    os.environ["INSPYRENET_CKPT_PATH"] = str(source_path)
    print("InSPyReNet 权重已准备, 不写入主方法代码:", source_path)

ensure_ceg_attestation_key()
prepare_inspyrenet_weight()

print("repo_root =", REPO_ROOT)
print("drive_root =", DRIVE_ROOT)
print("workspace =", PILOT_WORKSPACE_ROOT)


In [ ]:
# 3. 从 Google Drive 加载前序产物, 并检查图像生成输入和命令文件
# 若历史命令文件采用旧命名, 这里只做文件名迁移, 不修改正式 manifests。
CONFIG_ROOT = PILOT_WORKSPACE_ROOT / "configs"
if not COMMAND_FILE.exists() and CONFIG_ROOT.exists():
    command_candidates = sorted(CONFIG_ROOT.glob("*external_backend_command.draft.json"))
    if len(command_candidates) == 1:
        shutil.copyfile(command_candidates[0], COMMAND_FILE)
        print("已迁移 backend 命令文件:", command_candidates[0], "=>", COMMAND_FILE)

required_previous_artifacts = [
    PILOT_WORKSPACE_ROOT / "pilot_p0_input_freeze_report.json",
    PILOT_WORKSPACE_ROOT / "pilot_image_generation_launch_plan_report.json",
]
required_generation_inputs = [
    PROMPT_PLAN,
    MODEL_CONFIG,
]
if not APPLY_EXTERNAL_COMMAND_FROM_NOTEBOOK:
    required_generation_inputs.append(COMMAND_FILE)
required_paths = required_previous_artifacts + required_generation_inputs
missing_paths = []
for path in required_paths:
    exists = path.exists()
    print(("OK" if exists else "MISSING"), path)
    if not exists:
        missing_paths.append(path)

if missing_paths and STRICT_GOOGLE_DRIVE_PREFLIGHT:
    message = "Google Drive 工作区缺少图像生成前置文件。请先把前序产物结果保存到 PILOT_WORKSPACE_ROOT。
"
    message += "
".join(str(path) for path in missing_paths)
    raise FileNotFoundError(message)

if COMMAND_FILE.exists():
    payload = json.loads(COMMAND_FILE.read_text(encoding="utf-8-sig"))
    print("command_file_status =", payload.get("manifest_status"))
    print("has_external_command =", "external_command" in payload)
    print("has_placeholder =", "external_command_placeholder" in payload)


In [ ]:
# 4. 可选: 将真实外部 backend 命令写入命令文件
# 注意: 不要把 HF_TOKEN 或其他密钥写入 EXTERNAL_COMMAND。
if APPLY_EXTERNAL_COMMAND_FROM_NOTEBOOK:
    apply_cmd = [
        sys.executable,
        "scripts/apply_pilot_image_generation_backend_command.py",
        "--command-file",
        str(COMMAND_FILE),
        "--validation-report",
        str(COMMAND_VALIDATION_REPORT),
        "--external-command-json",
        json.dumps(EXTERNAL_COMMAND, ensure_ascii=False),
        "--require-ready",
    ]
    print("运行:", " ".join(apply_cmd))
    subprocess.run(apply_cmd, check=True)
else:
    print("跳过命令文件写入。请确认 COMMAND_FILE 已由你手工替换为 external_command。")


In [ ]:
# 5. 校验外部 backend 命令文件
validate_command = [
    sys.executable,
    "scripts/validate_pilot_image_generation_backend_command.py",
    "--command-file",
    str(COMMAND_FILE),
    "--out",
    str(COMMAND_VALIDATION_REPORT),
]
if REQUIRE_BACKEND_COMMAND_READY:
    validate_command.append("--require-ready")
print("运行:", " ".join(validate_command))
subprocess.run(validate_command, check=True)


In [ ]:
# 6. 执行图像生成包装入口
# 该步骤会调用 COMMAND_FILE 中的真实外部 SD / watermark backend。
# 如果 RUN_IMAGE_GENERATION_OUTPUTS 为 False, 本 cell 只打印应执行的命令。
image_generation_cmd = [
    sys.executable,
    "scripts/run_pilot_image_generation_backend.py",
    "--prompt-plan",
    str(PROMPT_PLAN),
    "--out",
    str(IMAGE_OUTPUT_ROOT),
    "--model-config",
    str(MODEL_CONFIG),
    "--external-command-json-file",
    str(COMMAND_FILE),
    "--require-pass",
]
print("图像生成命令:")
print(" ".join(image_generation_cmd))
if RUN_IMAGE_GENERATION_OUTPUTS:
    subprocess.run(image_generation_cmd, check=True)
else:
    print("RUN_IMAGE_GENERATION_OUTPUTS = False, 已跳过真实 GPU 执行。")


In [ ]:
# 7. 验收图像生成输出
# 只有该命令通过, 才能认为图像生成产物已完成。
image_generation_acceptance = [
    sys.executable,
    "scripts/validate_pilot_image_generation_outputs.py",
    "--output-root",
    str(IMAGE_OUTPUT_ROOT),
    "--out",
    str(PILOT_WORKSPACE_ROOT / "pilot_image_generation_output_acceptance_report.json"),
    "--require-pass",
]
print("运行:", " ".join(image_generation_acceptance))
if RUN_IMAGE_GENERATION_OUTPUTS:
    subprocess.run(image_generation_acceptance, check=True)
else:
    print("RUN_IMAGE_GENERATION_OUTPUTS = False, 已跳过图像生成通过性验收。")


In [ ]:
# 8. 刷新阶段进度, 并生成图像生成后的接续计划
summary_cmd = [
    sys.executable,
    "scripts/build_pilot_stage_progress_summary.py",
    "--workspace",
    str(PILOT_WORKSPACE_ROOT),
]
print("运行:", " ".join(summary_cmd))
subprocess.run(summary_cmd, check=True)

if BUILD_IMAGE_GENERATION_RESUME_PLAN:
    resume_cmd = [
        sys.executable,
        "scripts/build_pilot_image_generation_resume_plan.py",
        "--workspace",
        str(PILOT_WORKSPACE_ROOT),
    ]
    print("运行:", " ".join(resume_cmd))
    subprocess.run(resume_cmd, check=True)


In [ ]:
# 9. 归档图像生成产物到 Google Drive
# 该归档包只收集已经由仓库脚本生成和验收的文件, 不手写正式 manifests。
if ARCHIVE_IMAGE_GENERATION_OUTPUTS and RUN_IMAGE_GENERATION_OUTPUTS:
    ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = ARCHIVE_PATH
    include_paths = [
        IMAGE_OUTPUT_ROOT,
        PILOT_WORKSPACE_ROOT / "pilot_image_generation_output_acceptance_report.json",
        PILOT_WORKSPACE_ROOT / "pilot_stage_progress_summary.json",
        PILOT_WORKSPACE_ROOT / "pilot_stage_progress_summary.md",
        COMMAND_VALIDATION_REPORT,
    ]
    with ZipFile(archive_path, "w", compression=ZIP_DEFLATED) as archive:
        for item in include_paths:
            if item.is_dir():
                for file_path in item.rglob("*"):
                    if file_path.is_file():
                        archive.write(file_path, file_path.relative_to(PILOT_WORKSPACE_ROOT))
            elif item.is_file():
                archive.write(item, item.relative_to(PILOT_WORKSPACE_ROOT))
    print("已写出图像生成产物归档到 Google Drive CEG 目录:", archive_path)
else:
    print("未执行图像生成产物归档。若需要归档, 请确认 RUN_IMAGE_GENERATION_OUTPUTS 和 ARCHIVE_IMAGE_GENERATION_OUTPUTS 均为 True。")


In [ ]:
# 10. 显示关键结果路径
paths = {
    "image_generation_acceptance_report": PILOT_WORKSPACE_ROOT / "pilot_image_generation_output_acceptance_report.json",
    "stage_summary": PILOT_WORKSPACE_ROOT / "pilot_stage_progress_summary.json",
    "image_generation_resume_runbook": PILOT_WORKSPACE_ROOT / "gpu_handoff" / "image_generation_resume" / "pilot_image_generation_resume_runbook.md",
    "image_generation_archive": ARCHIVE_PATH,
    "image_pairs": IMAGE_OUTPUT_ROOT / "image_pairs.json",
    "clean_dir": IMAGE_OUTPUT_ROOT / "clean",
    "watermarked_dir": IMAGE_OUTPUT_ROOT / "watermarked",
}
for name, path in paths.items():
    print(name, "=", path, "exists=", path.exists())
